# Feature Creation

This notebook combines all of the features we have created based on the relevant activity patters.

This code takes in the activity log and grade files for both years and combines them into 1 dataframe with the created features. All of the created features are saved into a CSV file as training data for our model.


In [ ]:
#Needed data: activity_log_A_parandatud.csv; activity_log_B_parandatud.csv; grades_A.csv; grades_B.csv

#comment: activity_log_A_parandatud.csv and activity_log_B_parandatud.csv is different from activity_log_A.csv and activity_log_b.csv by some manually fixed typos.

from google.colab import files

files.upload();

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

## Required data

In [ ]:
# The provided activity datasets contain mistakes
# Found mistakes are fixed in this function
def fix_mistakes_in_activity_data(activities):
  activities["Activity"] = activities["Activity"].replace('Viewed the first final exam tasks after the exam in Estonian', 'Viewed first final exam tasks after the exam in Estonian')
  activities["Activity"] = activities["Activity"].replace('Viewed the first final exam tasks after the exam in English', 'Viewed first final exam tasks after the exam in English')

  activities["Activity"] = activities["Activity"].replace('Viewed the second final exam tasks after the exam in Estonian', 'Viewed second final exam tasks after the exam in Estonian')
  activities["Activity"] = activities["Activity"].replace('Viewed the second final exam tasks after the exam in English', 'Viewed second final exam tasks after the exam in English')

In [ ]:
# This function reads in activity logs and student grades for both years and combines them into 2 dataframes, one for grades and one for activities
def read_data():
  g1 = pd.read_csv('grades_A.csv')
  g2 = pd.read_csv('grades_B.csv')

  a1 = pd.read_csv('activity_log_A_parandatud.csv')
  a2 = pd.read_csv('activity_log_B_parandatud.csv')

  grades = pd.concat([g1, g2], ignore_index=True)
  activity = pd.concat([a1, a2], ignore_index=True)

  # Replacing missing data ("-" indicates that the student did not complete the task and, therefore, did not recieve a grade)
  grades['Grade'] = grades['Grade'].replace('-', "MI")
  grades['Total'] = grades['Total'].replace('-', -1)
  grades = grades.replace('-', 0)

  # Assigning numeric values to letter grades
  grade_mapping = {'A': 6, 'B': 5, 'C': 4, 'D': 3, 'E': 2, 'F': 1, 'MI': 0}
  grades['Grade_Num'] = grades['Grade'].map(grade_mapping)

  return activity, grades

# This function creates dictionaries of the deadlines for years A and B
def deadlines():
  deadlines_A = {}
  deadlines_B = {}

  deadlines_A['homework'] = [30, 37, 44, 51, 61, 68, 86, 93, 100, 107, 114, 117]
  deadlines_B['homework'] = [29, 36, 43, 50, 57, 64, 85, 92, 99, 106, 113, 120]

  deadlines_A['midterm'] = [68]
  deadlines_B['midterm'] = [68]

  deadlines_A['midterm_retake'] = [110, 124]
  deadlines_B['midterm_retake'] = [110, 124]

  deadlines_A['final'] = [110, 124]
  deadlines_B['final'] = [110, 124]

  deadlines_A['final_retake'] = [131]
  deadlines_B['final_retake'] = [131]

  return deadlines_A, deadlines_B

# This function combines sutudents grades from a tasks english and estonian versions into one.
# If a student did complete both of the activities, the higher grade of the two options is taken as their grade
def est_eng_sum():

  activity, grades = read_data()
  deadlines_A, deadlines_B = deadlines()

  for col in grades.columns[1:-3]:
      grades[col] = pd.to_numeric(grades[col])

  col_names = []
  for i in range(1, 13):
    grades[f'Homework {i}'] = grades[[f'Homework {i} (English version)', f'Homework {i} (Estonian version)']].max(axis=1)
  col_names.append(f'Homework {i}')

  grades['Midterm Exam'] = grades[['Midterm exam (English version)', 'Midterm exam (Estonian version)']].max(axis=1)
  grades['Midterm Exam Retake 1'] = grades[['Midterm exam retake (English version)', 'Midterm exam retake (Estonian version)']].max(axis=1)
  grades['Midterm Exam Retake 2'] = grades[['Midterm exam retake 2 (English version)', 'Midterm exam retake 2 (Estonian version)',]].max(axis=1)

  grades['Final Exam'] = grades[['First final exam (English version)', 'First final exam (Estonian version)', 'Second final exam (English version)', 'Second final exam (Estonian version)']].max(axis=1)
  grades['Final Exam Retake'] = grades[['Final exam retake (English version)', 'Final exam retake (Estonian version)']].max(axis=1)

  grades = grades.drop(columns=[f'Homework {i} (English version)' for i in range(1, 13)] + [f'Homework {i} (Estonian version)' for i in range(1, 13)])
  grades = grades.drop(columns=['Midterm exam (English version)', 'Midterm exam (Estonian version)', 'Midterm exam retake (English version)', 'Midterm exam retake (Estonian version)', 'Midterm exam retake 2 (English version)', 'Midterm exam retake 2 (Estonian version)'])
  grades = grades.drop(columns=['First final exam (English version)', 'First final exam (Estonian version)', 'Second final exam (English version)', 'Second final exam (Estonian version)', 'Final exam retake (English version)', 'Final exam retake (Estonian version)'])
  grades = grades.drop(columns=['Bonus'])

  return grades, activity

In [ ]:
d1 = pd.read_csv("activity_log_A_parandatud.csv")
d2 = pd.read_csv("activity_log_B_parandatud.csv")
d3 = pd.read_csv("grades_A.csv")
d4 = pd.read_csv("grades_B.csv")

activities = pd.concat([d1,d2])
grades = pd.concat([d3,d4])

# Additional mistakes are fixed
activities["Activity"] = activities["Activity"].replace('Viewed the first final exam tasks after the exam in Estonian', 'Viewed first final exam tasks after the exam in Estonian')
activities["Activity"] = activities["Activity"].replace('Viewed the first final exam tasks after the exam in English', 'Viewed first final exam tasks after the exam in English')

activities["Activity"] = activities["Activity"].replace('Viewed the second final exam tasks after the exam in Estonian', 'Viewed second final exam tasks after the exam in Estonian')
activities["Activity"] = activities["Activity"].replace('Viewed the second final exam tasks after the exam in English', 'Viewed second final exam tasks after the exam in English')

## Creating columns

In [ ]:
# replacing grades with a numeric value
def grade_numeric(grade):
  grades = ["-","F", "E", "D", "C", "B", "A"]
  return grades.index(grade)

# creating the students dataframe
students = pd.DataFrame({"Student":[], "Grade":[]})
for student in grades.iterrows():
   student_id = student[1][0]
   grade = student[1]["Grade"]
   new_row = {'Student': student_id, "Grade": grade_numeric(grade)}
   students = pd.concat([students, pd.DataFrame([new_row])], ignore_index=True)

<ipython-input-5-7d7f11e9622f>:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  student_id = student[1][0]


### How many times a student viewed a feedback on one of the homeworks

In [ ]:
# for features where different weeks were counted together
def replace_with_generic(str1):
  str1 = re.sub(r'\d+', 'N', str1)
  if "in English" in str1:
    str1 = str1.replace(" in English", "")
  if "in Estonian" in str1:
    str1 = str1.replace(" in Estonian", "")
  if "N" in str1:
    str1 = str1.replace(" N", "")
  return str1

In [ ]:
# counting feedback views
feedback_count = [0 for _, _ in students.iterrows()]
students["Feedback count"] = feedback_count

# adding the to the dataframe
for activity in activities.iterrows():
  student_id = activity[1]["Student"]
  activity_name = replace_with_generic(activity[1]["Activity"])
  if activity_name.startswith("Viewed the feedback"):
    students.loc[students["Student"] == student_id, "Feedback count"] += 1

### How many different activities did the student do for homework nr 6 and homework nr 11

Searching for correlations showed that these two homeworks were most correlated with the final grade. We figured that this is because these homeworks were during the midterms or finals, so students who were on track were able to focus more time on these homeworks.

In [ ]:
# creating the columns
HW6 = [0 for _, _ in students.iterrows()]
students["HW6"] = HW6
HW11 = [0 for _, _ in students.iterrows()]
students["HW11"] = HW11

# counting the activities where "Homework 6" or "Homework 11" were in the
for activity in activities.iterrows():
  student_id = activity[1]["Student"]
  activity_name = activity[1]["Activity"]
  if "Homework 6" in activity_name:
    students.loc[students["Student"] == student_id, "HW6"] += 1
  if "Homework 11" in activity_name:
    students.loc[students["Student"] == student_id, "HW11"] += 1

### How many days before the deadline the homeworks were open on average

In [ ]:
# the deadlines for each homework on year A and year B
hwA = [30, 37, 44, 51, 61, 68, 86, 93, 100, 107, 114, 117]
hwB = [29, 36, 43, 50, 57, 64, 85, 92, 99, 106, 113, 120]

# function to help differentiate between the students from first year and second year
def how_many_days_before(batch, hw, hwA, hwB, start):
    if batch == "A":
        return hwA[hw - 1] - start
    return hwB[hw - 1] - start

# counting
rows = []
for _, student in grades.iterrows():
    student_id = student["Student"]
    student_batch = student_id[0]
    # searching for all of the starts
    student_activities = activities[(activities["Student"] == student["Student"]) &
                                    (activities["Activity"].str.startswith("Started Homework"))]
    # finding the homework number for every homework and then using hwA and hwB to calculate how many days before the deadline
    # the start was - collecting these into a dictionary
    started = {}
    for _, activity in student_activities.iterrows():
        hw_number = int(activity["Activity"].split(" ")[-3])
        start = how_many_days_before(student_batch, hw_number, hwA, hwB, activity["Day"])
        if start > 0:
            started[f"HW{hw_number}"] = start
    # finding the average for each student
    if started:
        mean_value = sum(started.values()) / len(started)
        grade = grade_numeric(student["Grade"])

        rows.append({
            "Student": student_id,
            "First_opened_mean_HW": mean_value
        })

# addind the feature to the dataframe
students2 = pd.DataFrame(rows)
students = students.merge(students2, on="Student", how='left')

### How many time a student watched lectures online and preference (whether more were watched live online or more recordings)

In [ ]:
import numpy as np

# creating the columns
online_lectures = [0 for _, _ in students.iterrows()]
students["Online"] = online_lectures
recordings = [0 for _, _ in students.iterrows()]
students["Recording"] = recordings

# counting every recording and joining
for activity in activities.iterrows():
  student_id = activity[1]["Student"]
  activity_name = activity[1]["Activity"]
  if activity_name.startswith("Joined Week"):
    students.loc[students["Student"] == student_id, "Online"] += 1
  if activity_name.startswith("Watched Week"):
    students.loc[students["Student"] == student_id, "Recording"] += 1

# adding a preference column
students["Preference"] = np.where(students['Online'] > students['Recording'], 0, 1)

# the recordings are counted again later on
students = students.drop(columns=["Recording"])

### How many times did the student both join the lecture live and watch the recording afterwards

In [ ]:
# searching for every time a student watched a recording or joined live
activities['Online_week'] = [int(activ[12]) if activ.startswith("Joined Week") else 0 for activ in activities['Activity']]
activities['Recording_week'] = [int(activ[13]) if activ.startswith("Watched Week") else 0 for activ in activities['Activity']]

# finding the intersection between these two columns
both_counts = (
    activities.groupby('Student').apply(lambda x: len(set(x['Online_week']).intersection(set(x['Recording_week'])))).reset_index(name='Both')
)

# adding the feature to the dataframe
students["Both"] = both_counts["Both"]

<ipython-input-11-084eeb6d6c0b>:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: len(set(x['Online_week']).intersection(set(x['Recording_week']))))


In [ ]:
# intermediate check
students.head()

,Student,Grade,Feedback count,HW6,HW11,First_opened_mean_HW,Online,Preference,Both
0,A001,4.0,24,3,6,3.333333,1,0,1
1,A002,6.0,51,9,4,5.083333,6,0,1
2,A003,0.0,3,0,0,7.750000,1,1,1
3,A004,2.0,53,4,8,5.909091,1,1,2
4,A005,6.0,124,7,9,8.416667,4,0,2


### How many times did a student perform relevant activities over the whole semester

The activities were selected based on their correlation with a higher grade and feature importance (in testing).

In [ ]:
deadlines_A, deadlines_B = deadlines()
activity, grades = read_data()
students_list = grades["Student"]

In [ ]:
# Selecting only rows with target activites for more efficient processing
target_activities = [
  'Viewed Week \d+ lecture slides',
  'Viewed Week \d+ study materials in English',
  'Viewed Week \d+ study materials in Estonian',
  'Watched Week \d+ lecture recording',
  "Watched past years' Week \d+ lecture recording",
  'Viewed sample solutions of Homework \d+',
  'Viewed sample solutions of the midterm exam',
  'Viewed sample solutions of the final exam',
  "Viewed past years' exams"
]

filtered_activities = activity[
    activity.apply(lambda row: any(
        row.astype(str).str.contains('|'.join(target_activities), regex=True).any()
        for col in row.index), axis=1)
]


target_activities_15 = [
    ("Viewed Week", "") # combines the viewing count of lecture slides and study materials for each week
]

target_activities_1 = [
    ("Viewed sample solutions of Homework", ""), # counts the number of times a student viewed their homework (combines all homeworks)
    ("Viewed past years' exam", ""),
    ("Viewed sample solutions", ""), # combines the count of viewing the midterm and final examl sample solutions
    ("Watched", "") # combines lecture and lecture recording view count (all lectures combined)
]

activity_counts_dict = {}

for student in students_list:

  student_activities = filtered_activities.loc[filtered_activities['Student'] == student] # Get the activities for the current student

  student_activity_count = {}

  for week in range(1, 16):
    for a_prefix, a_suffix in target_activities_15:
        activity_name = f"{a_prefix} {week} {a_suffix}".strip()
        activity_occurrences = student_activities['Activity'].astype(str).str.contains(activity_name, case=False, regex=True).sum()
        student_activity_count["Viewed_lecture_" + str(week) + "_materials"] = activity_occurrences

  for a_prefix, _ in target_activities_1:
    activity_name = f"{a_prefix}".strip()
    activity_occurrences = student_activities['Activity'].astype(str).str.contains(activity_name, case=False, regex=True).sum()
    student_activity_count[activity_name] = activity_occurrences

  activity_counts_dict[student] = student_activity_count


activity_count_df = pd.DataFrame.from_dict(activity_counts_dict, orient='index')

activity_count_df = activity_count_df.reset_index()
activity_count_df = activity_count_df.rename(columns={'index': 'Student'})

# Adding the features to the main dataframe
students = students.merge(activity_count_df, on="Student", how='left')

In [ ]:
activity_count_df.head()

,Student,Viewed_lecture_1_materials,Viewed_lecture_2_materials,Viewed_lecture_3_materials,Viewed_lecture_4_materials,Viewed_lecture_5_materials,Viewed_lecture_6_materials,Viewed_lecture_7_materials,Viewed_lecture_8_materials,Viewed_lecture_9_materials,Viewed_lecture_10_materials,Viewed_lecture_11_materials,Viewed_lecture_12_materials,Viewed_lecture_13_materials,Viewed_lecture_14_materials,Viewed_lecture_15_materials,Viewed sample solutions of Homework,Viewed past years' exam,Viewed sample solutions,Watched
0,A001,41,5,9,8,8,8,9,10,12,6,10,8,7,3,1,16,0,16,0
1,A002,37,8,9,5,6,4,7,8,8,5,10,9,4,3,3,2,1,2,3
2,A003,6,7,3,3,1,1,0,0,0,0,0,0,0,0,0,0,0,0,7
3,A004,59,11,8,13,8,4,6,11,12,10,9,8,7,4,5,7,1,8,17
4,A005,88,5,8,9,8,8,7,4,5,11,14,16,14,17,11,13,3,19,3


### How many total activities did each student perform each week

The beginning and end day of each week is calculated based on homework deadlines. The first homework should be on the Sunday of the second week. The first few days in the beginning of the course before the start of week 1 are also considered as days of week one, so week one is a few days longer than the rest of the weeks.

The activities are calculated separately for years A and B, because the homework deadlines are on separate days.

In [ ]:
# This function counts the number of activities performed by each student for each week
def count_weekly_activity(activity_ab, deadlines):

  student_list_ab = activity_ab["Student"].unique()

  weekly_activity_df_ab = pd.DataFrame(
      index=student_list_ab,
      columns=[f"Total_activity_week_{week}" for week in range(1, 19)]
  ).fillna(0)

  week_start = 1
  week_end = deadlines["homework"][0] % 7 + 7

  for week in range(1, 19):
    week_activity = activity_ab[(activity_ab["Day"] <= week_end) & (activity_ab["Day"] >= week_start)]

    student_counts = week_activity["Student"].value_counts()
    weekly_activity_df_ab[f"Total_activity_week_{week}"] = (
          weekly_activity_df_ab[f"Total_activity_week_{week}"].add(student_counts, fill_value=0)
      )

    week_start = week_end + 1
    week_end += 7

  return weekly_activity_df_ab

activity_a = activity[activity["Student"].str.contains("A")]
activity_b = activity[activity["Student"].str.contains("B")]
deadlines_A, deadlines_B = deadlines()

weekly_activity_df_a = count_weekly_activity(activity_a, deadlines_A)
weekly_activity_df_b = count_weekly_activity(activity_b, deadlines_B)

weekly_activity_df = pd.concat([weekly_activity_df_a, weekly_activity_df_b], axis=0)
weekly_activity_df = weekly_activity_df.reset_index()
weekly_activity_df = weekly_activity_df.rename(columns={'index': 'Student'})

# Adding the features to the main dataframe
students = students.merge(weekly_activity_df, on="Student", how='left')

<ipython-input-16-09a976347af1>:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)
<ipython-input-16-09a976347af1>:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)


In [ ]:
weekly_activity_df

,Student,Total_activity_week_1,Total_activity_week_2,Total_activity_week_3,Total_activity_week_4,Total_activity_week_5,Total_activity_week_6,Total_activity_week_7,Total_activity_week_8,Total_activity_week_9,Total_activity_week_10,Total_activity_week_11,Total_activity_week_12,Total_activity_week_13,Total_activity_week_14,Total_activity_week_15,Total_activity_week_16,Total_activity_week_17,Total_activity_week_18
0,A020,4.0,0.0,5.0,6.0,5.0,4.0,2.0,5.0,6.0,25.0,2.0,3.0,5.0,11.0,7.0,51.0,50.0,8.0
1,A048,7.0,5.0,3.0,15.0,20.0,5.0,10.0,6.0,26.0,43.0,4.0,1.0,18.0,30.0,0.0,2.0,106.0,99.0
2,A005,5.0,5.0,8.0,14.0,22.0,12.0,10.0,19.0,4.0,45.0,14.0,20.0,28.0,25.0,46.0,70.0,51.0,1.0
3,A003,4.0,8.0,2.0,8.0,11.0,7.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,A035,7.0,2.0,13.0,1.0,14.0,10.0,12.0,8.0,20.0,35.0,4.0,1.0,3.0,3.0,2.0,7.0,25.0,93.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220,B058,0.0,0.0,6.0,0.0,9.0,3.0,3.0,4.0,6.0,2.0,0.0,2.0,0.0,5.0,0.0,2.0,3.0,1.0
221,B033,0.0,0.0,0.0,7.0,17.0,13.0,9.0,8.0,15.0,13.0,2.0,10.0,2.0,0.0,0.0,3.0,1.0,64.0
222,B036,0.0,0.0,0.0,1.0,7.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
223,B075,0.0,0.0,0.0,7.0,8.0,6.0,3.0,12.0,12.0,6.0,5.0,7.0,0.0,3.0,13.0,10.0,4.0,12.0


### How many days before the deadline student started and submitted their homework

In [ ]:
# This function makes 12 columns called 'start_before_HW{i}' and 12 columns called 'submit_before_HW{i}'

#Feature start before HW N deadline shows how many days before deadline did student start HW
# when student started homework on the last day, then value = 0
# when student did not start HW at all, then value = -1

#Feature submit before HW N deadline shows how many days before deadline did student submit HW
# when student submitted homework on the last day, then value = 0
# when student did not submit HW at all, then value = -1

def create_submission_columns(activity, deadlines_A, deadlines_B, start):

  if start:
    a = "Started Homework"
    c = "start_before"
  else:
    a = "Submitted Homework"
    c = "submit_before"

  students = activity['Student'].unique()
  result = pd.DataFrame(students, columns=['Student'])

  # Looping through each homework
  for i in range(1, 13):
      col_name = f'{c}_HW{i}'
      result[col_name] = -1  # when student did not start/submit HW at all

      for student in students:
          student_activities = activity[(activity['Student'] == student) &
                                        (activity['Activity'].str.contains(f'{a} {i}'))]

          if student_activities.empty:
              continue

          # Getting the earliest starting/submission date
          earliest_day = student_activities['Day'].min()

          # Determining the deadline based on the student's first letter
          if student[0] == 'A':
              deadline = deadlines_A[i - 1]
          elif student[0] == 'B':
              deadline = deadlines_B[i - 1]
          else:
              continue

          # Calculating days before deadline and assigning to the result DataFrame
          result.loc[result['Student'] == student, col_name] = deadline - earliest_day

  return result

submitted_HW = create_submission_columns(activity, deadlines_A['homework'], deadlines_B['homework'], False)
started_HW = create_submission_columns(activity, deadlines_A['homework'], deadlines_B['homework'], True)

students = students.merge(started_HW, on="Student", how='left')
students = students.merge(submitted_HW, on="Student", how='left')

### How many days before the homework deadlines did each student on average start/ submit their homework

In [ ]:
started_hw_si =  started_HW.copy()
submitted_hw_si =  submitted_HW.copy()

started_hw_si.set_index('Student', inplace=True)
submitted_hw_si.set_index('Student', inplace=True)

start_submit_avg = pd.DataFrame(
    index=started_hw_si.index,
    columns=["start_average", "submit_average"]
).fillna(0)

start_submit_avg["start_average"] = started_hw_si.mean(axis=1)
start_submit_avg["submit_average"] = submitted_hw_si.mean(axis=1)

start_submit_avg = start_submit_avg.reset_index()
start_submit_avg = start_submit_avg.rename(columns={'index': 'Student'})
students = students.merge(start_submit_avg, on="Student", how='left')

<ipython-input-19-957676ffc889>:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)


In [ ]:
start_submit_avg

,Student,start_average,submit_average
0,A020,7.750000,0.666667
1,A048,5.416667,-0.500000
2,A005,8.416667,1.833333
3,A003,1.916667,-0.750000
4,A035,3.833333,1.166667
...,...,...,...
220,B058,4.000000,0.166667
221,B033,-0.333333,-0.416667
222,B036,-0.333333,-1.000000
223,B075,4.500000,-0.166667


### Did student start HW on the last day and on the last hours? (Start time HW N)

In [ ]:
#This function makes 12 new columns called 'start_time_HW{i}'
# so every column/feature for every homework

# This feature indicates how many hours before the deadline did the student start doing the homework.
# The students who started homework more than 24h before the HW are treated as same.
# From this feature distinctly come out students who start HW on the last day and especially if the student started HW in the last hours.

# If student started HW before the last day, then the value = 25.
# If on the last day at 01:00 then value = 23, at 02:00 value = 22, …, at 10:00 value = 14,
# …, at 18:00 value = 6, …, at 23:00 value = 1, at 00:00 value = 0,
# after that (on a late day) value = 0,
# when didn't start HW at all value  = -1.

def create_start_time_columns(activity, deadlines_A, deadlines_B):

    students = activity['Student'].unique()
    result = pd.DataFrame(students, columns=['Student'])

    # Looping through each homework
    for i in range(1, 13):
        col_name = f'start_time_HW{i}'
        result[col_name] = -1  # when student did not start HW at all

        for student in students:
            # Getting the relevant rows for this student and homework
            student_activities = activity[(activity['Student'] == student) &
                                          (activity['Activity'].str.contains(f'Started Homework {i}'))]

            # If there are no relevant rows, continuing to the next student
            if student_activities.empty:
                continue

            # Getting the earliest submission for this homework
            earliest_start = student_activities.loc[student_activities['Day'].idxmin()]
            start_day = earliest_start['Day']
            start_time = earliest_start['Time']

            # Determining the homeworks deadline based on the student's first letter
            if student[0] == 'A':
                deadline_day = deadlines_A[i - 1]
            elif student[0] == 'B':
                deadline_day = deadlines_B[i - 1]
            else:
                continue

            if start_day < deadline_day:
                result.loc[result['Student'] == student, col_name] = 25 #when student started HW before the deadline day
            elif start_day == deadline_day:
                # Calculating hours before midnight of deadline day
                hours_before_midnight = 24 - (start_time.hour + start_time.minute / 60 + start_time.second / 3600)
                result.loc[result['Student'] == student, col_name] = hours_before_midnight #when student started on the deadline day
            else:
                result.loc[result['Student'] == student, col_name] = 0  # when student started HW after deadline day

    return result

activity, grades = read_data()
activity['Time'] = pd.to_datetime(activity['Time'], format='%H:%M:%S').dt.time
deadlines_A, deadlines_B = deadlines()

start_time_HW = create_start_time_columns(activity, deadlines_A['homework'], deadlines_B['homework'])

students = students.merge(start_time_HW, on = 'Student', how = 'left')

<ipython-input-21-441e7041834d>:51: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4.898055555555555' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  result.loc[result['Student'] == student, col_name] = hours_before_midnight #when student started on the deadline day
<ipython-input-21-441e7041834d>:51: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '5.280833333333334' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  result.loc[result['Student'] == student, col_name] = hours_before_midnight #when student started on the deadline day
<ipython-input-21-441e7041834d>:51: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.5402777777777779' has dtype incompatible with int64, 

### Did the student study on the day of (retake) exam at 00:00-05:00?

In [ ]:
# This function makes 2 new columns called 'study_night_final2' and 'study_night_final_retake'.
# If the student did any kind of activity on the day of (retake) exam at 00:00-05:00 which is the night before (retake) exam then the value = 1.
# If there was no activity done at this timeframe then the value = 0.

from datetime import time

exam_names = ['midterm', 'final', 'final2', 'final_retake'] # studying in the night before midterm or first final exam was not correlated with the grades
exam_dates = [68, 110, 124, 131] # same dates for both years (A and B students)

#activity['Time'] = pd.to_datetime(activity['Time'], format='%H:%M:%S').dt.time

for i in [2, 3]: # using only significant features: final2 and final_retake
    exam = exam_names[i]
    day = exam_dates[i]
    students[f'study_night_{exam}'] = students.apply(
        lambda row: 1 if any( # when student did an activity at this timeframe
            (activity['Student'] == row['Student']) &
            (activity['Day'] == day) &
            (activity['Time'] >= time(0, 0, 0)) &
            (activity['Time'] <= time(5, 0, 0))
        ) else 0, # when student did not do any activities at this timeframe
        axis=1
    )

# Downloading the final dataframe with all the features

In [ ]:
students = students.fillna(0)

In [ ]:
students.to_csv('all_features.csv', index=False)